# Hypergraph Analysis

In [1]:
import os
import pandas as pd
import hypernetx as hnx
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import hypergraph.hyperGraphs as hnb
import hypergraph.visualization as viz

import optim_help.utilities as ut

## Load and pre-process selected hyperedges data

### Hyperedges from CLIQUES EXPANSION - POLY & NPN

In [3]:
sel_he_base_path = '../DATA/selected_HYPEREDGES_data/'

In [4]:
# Hyperedges selected from the polychoric correlation dyadic graph's cliques
cliques_POLY_path = sel_he_base_path + 'POLY'
cliques_POLY_paths = [
    cliques_POLY_path + '/' + f
    for f in os.listdir(cliques_POLY_path)]

# Hyperedges selected from the nonparanormal (NPN) correlation dyadic graph's cliques
cliques_NPN_path = sel_he_base_path + 'NPN'
cliques_NPN_paths = [
    cliques_NPN_path + '/' + f
    for f in os.listdir(cliques_NPN_path)]

# df cols : order, label, omega, node1, node2 ... node6

In [6]:
cliques_NPN_paths

['../DATA/selected_HYPEREDGES_data/NPN/ANBP_selected_he_NPN.csv',
 '../DATA/selected_HYPEREDGES_data/NPN/ANR_selected_he_NPN.csv',
 '../DATA/selected_HYPEREDGES_data/NPN/BED_OSFED_selected_he_NPN.csv',
 '../DATA/selected_HYPEREDGES_data/NPN/BN_selected_he_NPN.csv']

In [7]:
# FROM POLYCHORIC CORRELATION 
syn_POLY = {}
red_POLY = {}
syn_properties_POLY = {}
red_properties_POLY = {}

for hg_path in cliques_POLY_paths:
    diag = hg_path.split('/')[-1].split('_')[0]
    # manually set BED to BED_OSFED
    if diag == 'BED':
        diag = 'BED_OSFED'
    he, he_prop = hnb.preproc_hyperedges(hg_path, 'clique_exp')
    # divide syn and red
    syn_properties_POLY[diag] = {k: v for k, v in he_prop.items() if v.get("label") == "synergy"}
    red_properties_POLY[diag] =  {k: v for k, v in he_prop.items() if v.get("label") == "redundancy"}
    # now divide also the he
    syn_POLY[diag] = {k: he[k] for k in syn_properties_POLY[diag].keys() if k in he}
    red_POLY[diag] =  {k: he[k] for k in red_properties_POLY[diag].keys() if k in he}

In [8]:
# FROM NONPARANORMAL (NPN) CORRELATION 
syn_NPN = {}
red_NPN = {}
syn_properties_NPN = {}
red_properties_NPN = {}

for hg_path in cliques_NPN_paths:
    diag = hg_path.split('/')[-1].split('_')[0]
    # manually set BED to BED_OSFED
    if diag == 'BED':
        diag = 'BED_OSFED'
    he, he_prop = hnb.preproc_hyperedges(hg_path, 'clique_exp')
    # divide syn and red
    syn_properties_NPN[diag] = {k: v for k, v in he_prop.items() if v.get("label") == "synergy"}
    red_properties_NPN[diag] =  {k: v for k, v in he_prop.items() if v.get("label") == "redundancy"}
    # now divide also the he
    syn_NPN[diag] = {k: he[k] for k in syn_properties_NPN[diag].keys() if k in he}
    red_NPN[diag] =  {k: he[k] for k in red_properties_NPN[diag].keys() if k in he}

In [10]:
red_NPN['ANBP']

{'9-55-59': [9, 55, 59], '12-19-32-55-62': [12, 19, 32, 55, 62]}

### Hyperedges from sub-SCALES combination 

In [11]:
# Hyperedges selected from the combinations among items in the same sub-scale
scales_path = sel_he_base_path + 'SCALES'
scales_paths = [
    scales_path + '/' + f
    for f in os.listdir(scales_path)]

In [13]:
#pd.read_csv('../DATA/selected_HYPEREDGES_data/selected_df_scales_FINAL_INTRA/ANR_selected_df_scales.csv')
scales_paths

['../DATA/selected_HYPEREDGES_data/SCALES/ANBP_selected_df_scales.csv',
 '../DATA/selected_HYPEREDGES_data/SCALES/ANR_selected_df_scales.csv',
 '../DATA/selected_HYPEREDGES_data/SCALES/BED_OSFED_selected_df_scales.csv',
 '../DATA/selected_HYPEREDGES_data/SCALES/BN_selected_df_scales.csv']

In [14]:
syn_SCALES = {}
red_SCALES = {}
syn_properties_SCALES = {}
red_properties_SCALES = {}

for hg_path in scales_paths:
    diag = hg_path.split('/')[-1].split('_')[0]
    # manually set BED to BED_OSFED
    if diag == 'BED':
        diag = 'BED_OSFED'
    he, he_prop = hnb.preproc_hyperedges(hg_path, 'cliques-exp')
    # divide syn and red
    syn_properties_SCALES[diag] = {k: v for k, v in he_prop.items() if v.get("label") == "synergy"}
    red_properties_SCALES[diag] =  {k: v for k, v in he_prop.items() if v.get("label") == "redundancy"}
    # now divide also the he
    syn_SCALES[diag] = {k: he[k] for k in syn_properties_SCALES[diag].keys() if k in he}
    red_SCALES[diag] =  {k: he[k] for k in red_properties_SCALES[diag].keys() if k in he}

In [13]:
#he # ('1-2-7-19'): {1, 2, 7, 19}
#he_prop # ('1-2-7-19'): {'omega': ...

#### Merge hyperedges/multiplets: POLY, NPN; SCALES.

In [15]:
merged_SYN = {}
merged_RED = {}
merged_prop_SYN = {}
merged_prop_RED = {}

for diag in syn_POLY:
    merged_SYN[diag] = syn_POLY[diag] | syn_NPN[diag] | syn_SCALES[diag]
    merged_prop_SYN[diag] = syn_properties_POLY[diag] | syn_properties_NPN[diag] | syn_properties_SCALES[diag]

    merged_RED[diag] = red_POLY[diag] | red_NPN[diag] | red_SCALES[diag] 
    merged_prop_RED[diag] = red_properties_POLY[diag] | red_properties_NPN[diag] | red_properties_SCALES[diag]

Intersections between scales/poly-npn.

In [23]:
for diag in syn_POLY:
    merged_poly_npn_SYN = syn_POLY[diag] | syn_NPN[diag]
    npol_scales_intersec_SYN = set(merged_poly_npn_SYN.keys()).intersection(set(syn_SCALES[diag].keys())) 
    print('Scales and POLY/NPN SYN intersection is', len(npol_scales_intersec_SYN))

print('\n')
for diag in red_POLY:    
    merged_poly_npn_RED = red_POLY[diag] | red_NPN[diag]
    npol_scales_intersec_RED = set(merged_poly_npn_RED.keys()).intersection(set(red_SCALES[diag].keys())) 
    print('Scales and POLY/NPN RED intersection is', len(npol_scales_intersec_RED))

Scales and POLY/NPN SYN intersection is 88
Scales and POLY/NPN SYN intersection is 83
Scales and POLY/NPN SYN intersection is 44
Scales and POLY/NPN SYN intersection is 77


Scales and POLY/NPN RED intersection is 1
Scales and POLY/NPN RED intersection is 14
Scales and POLY/NPN RED intersection is 7
Scales and POLY/NPN RED intersection is 4


Intersections between poly/npn.

In [16]:
npn_poly_intersec_SYN = {}
npn_poly_intersec_RED = {}

for diag in syn_POLY:
    print(diag)
    npn_poly_intersec_SYN[diag] = set(syn_POLY[diag].keys()).intersection(set(syn_NPN[diag].keys())) 
    npn_poly_intersec_RED[diag] = set(red_POLY[diag].keys()).intersection(set(red_NPN[diag].keys())) 
    print('SYN_poly:', len(syn_POLY[diag]), '- SYN_npn:', len(syn_NPN[diag]), '- SYN_intersec', len(npn_poly_intersec_SYN[diag]))
    print('RED_poly:', len(red_POLY[diag]), '- RED_npn:', len(red_NPN[diag]), '- RED_intersec', len(npn_poly_intersec_RED[diag]))
    
    print('SYN_union:', len(syn_POLY[diag] | syn_NPN[diag]))
    print('RED_union:', len(red_POLY[diag] | red_NPN[diag]))
    print('\n')

ANBP
SYN_poly: 573 - SYN_npn: 592 - SYN_intersec 344
RED_poly: 3 - RED_npn: 2 - RED_intersec 0
SYN_union: 821
RED_union: 5


ANR
SYN_poly: 624 - SYN_npn: 692 - SYN_intersec 167
RED_poly: 22 - RED_npn: 26 - RED_intersec 17
SYN_union: 1149
RED_union: 31


BED_OSFED
SYN_poly: 497 - SYN_npn: 669 - SYN_intersec 343
RED_poly: 18 - RED_npn: 22 - RED_intersec 8
SYN_union: 823
RED_union: 32


BN
SYN_poly: 491 - SYN_npn: 776 - SYN_intersec 228
RED_poly: 3 - RED_npn: 2 - RED_intersec 0
SYN_union: 1039
RED_union: 5




#### Merged hes counts.

#### FILTER (here not needed - already filtered by 0.1 during selection) & CHECK FOR SINGLETONS

In [20]:
merged_SYN_filtered, merged_prop_SYN_filtered = ut.filtering_he(merged_prop_SYN, merged_SYN, threshold=0.10)

ANBP 2086 0 2086 0
ANR 2215 0 2215 0
BED_OSFED 1970 0 1970 0
BN 1817 0 1817 0


In [21]:
merged_RED_filtered, merged_prop_RED_filtered = ut.filtering_he(merged_prop_RED, merged_RED, threshold=0.10)

ANBP 30 0 30 0
ANR 40 0 40 0
BED_OSFED 80 0 80 0
BN 20 0 20 0


In [31]:
for diag in syn_POLY:
    print(diag)
    print('Originals:', len(merged_prop_SYN[diag]), "SYN.", len(merged_prop_RED[diag]), "RED.")
    print('After further filtering', len(merged_prop_SYN_filtered[diag]), "SYN.", len(merged_prop_RED_filtered[diag]), "RED.")
    print('----------------------------------------')

ANBP
Originals: 2086 SYN. 30 RED.
After further filtering 2086 SYN. 30 RED.
----------------------------------------
ANR
Originals: 2215 SYN. 40 RED.
After further filtering 2215 SYN. 40 RED.
----------------------------------------
BED_OSFED
Originals: 1970 SYN. 80 RED.
After further filtering 1970 SYN. 80 RED.
----------------------------------------
BN
Originals: 1817 SYN. 20 RED.
After further filtering 1817 SYN. 20 RED.
----------------------------------------


In [32]:
for d, interactions in merged_prop_SYN_filtered.items():
    print(d)
    d_int = list(interactions.values())
    int_3 = [i['nodes_numeric'] for i in d_int if len(i['nodes_numeric']) == 3]
    int_4 = [i['nodes_numeric'] for i in d_int if len(i['nodes_numeric']) == 4]
    int_5 = [i['nodes_numeric'] for i in d_int if len(i['nodes_numeric']) == 5]
    print('3:', len(int_3), '; 4:', len(int_4), '; 5:', len(int_5))

ANBP
3: 111 ; 4: 1932 ; 5: 43
ANR
3: 8 ; 4: 1850 ; 5: 357
BED_OSFED
3: 190 ; 4: 1769 ; 5: 11
BN
3: 4 ; 4: 1255 ; 5: 558


In [33]:
for d, interactions in merged_prop_RED_filtered.items():
    print(d)
    d_int = list(interactions.values())
    int_3 = [i['nodes_numeric'] for i in d_int if len(i['nodes_numeric']) == 3]
    int_4 = [i['nodes_numeric'] for i in d_int if len(i['nodes_numeric']) == 4]
    int_5 = [i['nodes_numeric'] for i in d_int if len(i['nodes_numeric']) == 5]
    print('3:', len(int_3), '; 4:', len(int_4), '; 5:', len(int_5))

ANBP
3: 17 ; 4: 2 ; 5: 11
ANR
3: 31 ; 4: 2 ; 5: 7
BED_OSFED
3: 13 ; 4: 19 ; 5: 48
BN
3: 13 ; 4: 1 ; 5: 6


In [36]:
# skip dir creation if it already exist
out_dir_sm = "tables"            
os.makedirs(out_dir_sm, exist_ok=True)

with open("tables/summary_SELECTED_HE_filtered.txt", "w") as f:
    print('MERGED - FINAL --------------------------------', file = f)
    for diag in syn_POLY:
        print(diag, file = f)
        print(len(merged_prop_SYN_filtered[diag]), "synergy hyperedges", file = f)
        print(len(merged_prop_RED_filtered[diag]), "redundancy hyperedges", file = f)
        print('----------------------------------------', file = f)
    
    print('\n', file = f)
    print('\n', file = f)
    
    print('POLY ------------------------------------------', file = f)
    for diag, hes in syn_POLY.items():
        print(diag, file = f)
        print(len(syn_POLY[diag]), "synergy hyperedges", file = f)
        print(len(red_POLY[diag]), "redundancy hyperedges", file = f)
        print('----------------------------------------', file = f)
        
    print('\n', file = f)

    print('NPN ------------------------------------------', file = f)
    for diag, hes in syn_NPN.items():
        print(diag, file = f)
        print(len(syn_NPN[diag]), "synergy hyperedges", file = f)
        print(len(red_NPN[diag]), "redundancy hyperedges", file = f)
        print('----------------------------------------', file = f)
        
    print('\n', file = f)
    print('\n', file = f)

    print('SCALES ------------------------------------------', file = f)
    for diag, hes in syn_SCALES.items():
        print(diag, file = f)
        print(len(syn_SCALES[diag]), "synergy hyperedges", file = f)
        print(len(red_SCALES[diag]), "redundancy hyperedges", file = f)
        print('----------------------------------------', file = f)
    
    print('\n', file = f)

### Nodes: description

In [37]:
nodes_labels_path = '../DATA/EDI_det/edi3_items_fixed.csv' 
nodes_items = pd.read_csv(nodes_labels_path)

In [38]:
nodes_items

,desc,id
0,I eat sweets and carbohydrates without feeling...,1
1,I think that my stomach is too big.,2
2,I wish that I could return to the security of ...,3
3,I eat when I am upset.,4
4,I binge eat.,5
...,...,...
86,I would rather spend time by myself than with ...,87
87,Suffering makes you a better person.,88
88,I know there are people who love me.,89
89,I feel like I must hurt myself or others.,90


In [39]:
nodes_items_dict = dict(zip(nodes_items["id"], nodes_items["desc"]))

## Build the Multiplex HyperGraph

### Partition and Multiplex Construction 

**After all three validation stages:**

1. **Partition by sign:**
   - **Redundant hyperedges:** Ω > 0 (positive layer)
   - **Synergistic hyperedges:** Ω < 0 (negative layer)

2. **Multiplex construction:**
     - Hypergraph 1 (positive): Weighted hyperedges with weight = +|Ω|
         - a layer for each diagnosis
     - Hypergraph 2 (negative): Weighted hyperedges with weight = -|Ω|
         - a layer for each diagnosis
 
Aggregate across all diagnoses by summing hyperedge weights across layers

Result: Multiplex hypergraphs with diagnosis-specific and cross-diagnosis structures

Run this following cell to recompute the tables with the furtherly filtered hyperedges.

In [40]:
merged_SYN = merged_SYN_filtered
merged_prop_SYN = merged_prop_SYN_filtered

merged_RED = merged_RED_filtered
merged_prop_RED = merged_prop_RED_filtered

In [41]:
# check for singletons - no singoletti in neither SYN nor RED
for k, v in merged_RED.items():
    singletons = [e for e in v.values() if len(e) == 1]
    #print(singletons)

#### merged_SYN, merged_RED 
#### merged_prop_SYN, merged_prop_RED 

In [42]:
HG_multiplex = {}
for diag in merged_SYN:
    print(diag)
    HG_syn, HG_red = diag + '_SYN', diag + '_RED'
    HG_multiplex[HG_syn] = hnb.build_hypergraph_from_multiplets(merged_SYN[diag], merged_prop_SYN[diag], nodes_items_dict)
    HG_multiplex[HG_red] = hnb.build_hypergraph_from_multiplets(merged_RED[diag], merged_prop_RED[diag], nodes_items_dict)

ANBP
Missing nodes are: 0 set()
Missing nodes are: 71 {1, 3, 4, 6, 7, 8, 10, 11, 13, 14, 16, 17, 18, 20, 21, 23, 24, 25, 26, 27, 29, 30, 33, 35, 36, 37, 40, 41, 42, 43, 44, 47, 48, 49, 50, 51, 52, 53, 54, 56, 60, 61, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91}
ANR
Missing nodes are: 0 set()
Missing nodes are: 65 {1, 3, 4, 6, 8, 10, 11, 13, 14, 15, 17, 18, 20, 22, 23, 24, 27, 29, 30, 33, 34, 35, 36, 39, 40, 42, 43, 44, 47, 48, 52, 53, 54, 56, 57, 58, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91}
BED_OSFED
Missing nodes are: 0 set()
Missing nodes are: 66 {3, 6, 8, 10, 13, 14, 15, 17, 18, 20, 21, 22, 23, 24, 26, 27, 29, 30, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 48, 50, 51, 52, 54, 56, 57, 58, 60, 63, 64, 65, 66, 67, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91}
BN
Missing nodes are: 0 set()
Missing

In [43]:
HG_ANBP_SYN = HG_multiplex['ANBP_SYN']

In [44]:
HG_ANBP_SYN.edges.dataframe.head()

,weight,misc_properties
uid,,
3-35-39,-0.107673,"{'omega': -0.1076726411963058, 'label': 'syner..."
5-29-71,-0.268419,"{'omega': -0.2684185260693716, 'label': 'syner..."
6-14-35,-0.116951,"{'omega': -0.1169509284954966, 'label': 'syner..."
6-18-24,-0.101958,"{'omega': -0.1019583143981694, 'label': 'syner..."
6-22-35,-0.102736,"{'omega': -0.1027355043562199, 'label': 'syner..."


In [45]:
common_hes_SYN = set()
for name, hg in HG_multiplex.items():
    if 'SYN' in name:
        print(name, '--------------')
        hg_index = hg.edges.dataframe.index
        print('Singleton search', [a for a in hg_index if a == '71'])
        hg_index_one = [e for e in hg_index if len(e) == 1]
        hg_index = [e for e in hg_index if len(e) > 1] # and e != '71']
        common_hes_SYN = common_hes_SYN | set(hg_index)
        print(name, hg_index_one, len(hg.edges.dataframe))
print(len(common_hes_SYN))

common_hes_RED = set()
for name, hg in HG_multiplex.items():
    if 'RED' in name:
        hg_index = hg.edges.dataframe.index
        hg_index_one = [e for e in hg_index if len(e) == 1]
        hg_index = [e for e in hg_index if len(e) > 1]
        common_hes_RED = common_hes_RED | set(hg_index)
        print(name, hg_index_one, len(hg.edges.dataframe))
print(len(common_hes_RED))

ANBP_SYN --------------
Singleton search []
ANBP_SYN [] 2086
ANR_SYN --------------
Singleton search []
ANR_SYN [] 2215
BED_OSFED_SYN --------------
Singleton search []
BED_OSFED_SYN [] 1970
BN_SYN --------------
Singleton search []
BN_SYN [] 1817
5949
ANBP_RED ['1', '3', '4', '6', '7', '8'] 101
ANR_RED ['1', '3', '4', '6', '8'] 105
BED_OSFED_RED ['3', '6', '8'] 146
BN_RED ['1', '3', '6', '7', '8'] 96
209


In [46]:
HG_ANBP_SYN.nodes.dataframe

,weight,misc_properties
uid,,
3,1,{'desc': 'I wish that I could return to the se...
35,1,{'desc': 'The demands of adulthood are too gre...
39,1,{'desc': 'I feel happy that I am not a child a...
5,1,{'desc': 'I binge eat.'}
29,1,"{'desc': 'As a child, I tried very hard to avo..."
...,...,...
74,1,{'desc': 'I feel trapped in relationships.'}
38,1,{'desc': 'I think about bingeing (overeating).'}
28,1,{'desc': 'I have gone on eating binges where I...


### Sub-scales description

In [47]:
scales_desc = pd.DataFrame({
    "scale": ['-', 'A', 'B', 'BD', 'DT', 'ED', 'IA', 'ID', 'II', 'LSE', 'MF', 'P', 'PA'],
    "desc": ['-', 'Ascetism', 'Bulimia', 'Body Dissatisfaction', 'Drive for Thinness',
             'Emotional Disregulation', 'Interpersonal Alienation', 'Interoceptive Deficits',
             'Interpersonal Insecurity', 'Low Self Esteem', 'Maturity Fears',
             'Perfectionism', 'Personal Alienation']
})

In [48]:
scales_desc

,scale,desc
0,-,-
1,A,Ascetism
2,B,Bulimia
3,BD,Body Dissatisfaction
4,DT,Drive for Thinness
5,ED,Emotional Disregulation
6,IA,Interpersonal Alienation
7,ID,Interoceptive Deficits
8,II,Interpersonal Insecurity
9,LSE,Low Self Esteem


## NODES PROPERTIES

#### Paper: Multiplex measures for higher‑order networks (2024) - Lotito, Montresor, Battiston 

### WEIGHTED DEGREE

In [49]:
# layer is the diagnosis
wDeg_layers = {}
H_incS = {}
edges_layers = {}
for layer, HG in HG_multiplex.items():
    '''
    H_inc : (N,E) incidence matrix
    w     : (E,) edge weights
    k_w   : (N,) weighted degree
    k_norm: (N,) normalized weighted degree
    nodes : list of nodes
    edges : list of edges
    '''
    H_inc, w, k_w, k_norm, nodes, edges = hnb.weighted_degrees_normalized(HG)
    H_incS[layer] = H_inc
    edges_layers = edges
    wDeg_layers[layer] = pd.DataFrame({
                                        "node": nodes,
                                        "weighted_degree": k_w,
                                        "normalized_weighted_degree": k_norm
                                    })

In [50]:
wDeg_layers['ANR_SYN'].head(5)

,node,weighted_degree,normalized_weighted_degree
0,13,-39.048483,0.006114
1,38,-8.290081,0.001298
2,64,-12.875059,0.002016
3,89,-38.035279,0.005955
4,1,-233.933756,0.036628


In [51]:
M_norm = hnb.combine_layer_weighted_degrees(wDeg_layers, value="normalized_weighted_degree")

In [52]:
print(M_norm.loc['ANR_SYN'].loc[71])

0.0006763729131236449


In [53]:
M_norm

node,1,2,3,4,5,6,7,8,9,10,...,82,83,84,85,86,87,88,89,90,91
ANBP_SYN,0.017934,0.010290,0.021243,0.007000,0.006248,0.007209,0.004872,0.030639,0.002595,0.012318,...,0.014748,0.006752,0.018923,0.018435,0.019554,0.017461,0.006307,0.005099,0.003472,0.008792
ANBP_RED,0.000000,0.033552,0.000000,0.000000,0.022710,0.000000,0.000000,0.000000,0.086605,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ANR_SYN,0.036628,0.013895,0.020345,0.011311,0.004328,0.015154,0.008343,0.041445,0.003999,0.014241,...,0.011583,0.004011,0.016801,0.012754,0.012033,0.012973,0.002277,0.005955,0.002192,0.017497
ANR_RED,0.000000,0.067352,0.000000,0.000000,0.004469,0.000000,0.057268,0.000000,0.131377,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
BED_OSFED_SYN,0.013131,0.008113,0.017657,0.004306,0.001577,0.006401,0.005716,0.029208,0.001836,0.012174,...,0.010866,0.009297,0.029428,0.017805,0.014530,0.021787,0.007987,0.005313,0.002216,0.014103
BED_OSFED_RED,0.026612,0.035642,0.000000,0.020450,0.055445,0.000000,0.030447,0.000000,0.067403,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
BN_SYN,0.032940,0.024449,0.018306,0.008187,0.005143,0.033264,0.006451,0.030393,0.004806,0.021771,...,0.009599,0.005380,0.008016,0.008623,0.009823,0.010861,0.009807,0.023219,0.008568,0.018207
BN_RED,0.000000,0.047696,0.000000,0.011878,0.011878,0.000000,0.000000,0.000000,0.102192,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [54]:
nodes_items_dict = dict(zip(nodes_items['id'], nodes_items['desc']))

In [55]:
HG_desc = M_norm.apply(lambda row: ut.process_row_wn(row, nodes_items_dict), axis=1)

In **HG_desc**, nodes are sorted by weighted degree.

In [56]:
HG_desc

ANBP_SYN         {'row': 'ANBP_SYN', 'nodes': [40, 77, 44, 33, ...
ANBP_RED         {'row': 'ANBP_RED', 'nodes': [55, 62, 45, 19, ...
ANR_SYN          {'row': 'ANR_SYN', 'nodes': [8, 40, 1, 44, 33,...
ANR_RED          {'row': 'ANR_RED', 'nodes': [9, 32, 45, 49, 55...
BED_OSFED_SYN    {'row': 'BED_OSFED_SYN', 'nodes': [40, 77, 44,...
BED_OSFED_RED    {'row': 'BED_OSFED_RED', 'nodes': [62, 45, 9, ...
BN_SYN           {'row': 'BN_SYN', 'nodes': [6, 1, 8, 69, 24, 2...
BN_RED           {'row': 'BN_RED', 'nodes': [55, 62, 45, 9, 19,...
dtype: object

In [57]:
HG_desc.iloc[1]

{'row': 'ANBP_RED',
 'nodes': Index([55, 62, 45, 19, 12, 9, 31, 59, 2, 28, 5, 46, 38, 32, 15, 34, 57, 22, 39,
        58],
       dtype='int64', name='node'),
 'values': array([0.17256359, 0.14723365, 0.12031325, 0.09434717, 0.0928428 ,
        0.08660483, 0.05301958, 0.05166967, 0.03355245, 0.03086954,
        0.02270994, 0.01987267, 0.01915646, 0.01405421, 0.00736833,
        0.00736833, 0.00736833, 0.00636174, 0.00636174, 0.00636174])}

######  HG_desc is a series of dictionaries with keys row (layer, like ANBO:SYN), nodes (of he HG), values (the weighted degree)

#### Mapping item: scale (ex. x: DT)

- items_scales_dict maps nodes (as int - key) to the sub_scale they belong to (as str - value)
- scores are normalized (they sum up to 1)

In [58]:
sub_scales_path = '../DATA/EDI_det/sub_scales.csv' 
sub_scales = pd.read_csv(sub_scales_path)[['item', 'scales']]

scales_df_dict = {name: group for name, group in sub_scales.groupby("scales")}
items_by_scale = sub_scales.groupby("scales")["item"].apply(list).to_dict()
items_scales_dict = sub_scales.set_index("item")["scales"].to_dict()

In [59]:
scales_desc_dict = scales_desc.set_index("scale")["desc"].to_dict()

In [60]:
with open("tables/items_per_HG_filtered.txt", "w") as f:
    for row in HG_desc:
        print(row['row'], file = f)
        for j, v in zip(row['nodes'], row['values']):
            scale = items_scales_dict[j]
            scale_d = '(' + scale + '-' + scales_desc_dict[scale] + ')'
            print(j, round(v, 4), scale_d, ':', nodes_items_dict[j] + '\n', file = f)
        print('*************', file = f)

### NODES' BEHAVIOUR wrt SUB_SCALES

In [62]:
from collections import Counter

data = list(items_scales_dict.items())
# count occurrences of second element
counts = Counter(t[1] for t in data)
# convert to regular dict (optional)
scale_arity = dict(counts)

print(scale_arity)

{'DT': 7, 'BD': 10, 'MF': 8, 'B': 8, 'ID': 9, 'LSE': 6, 'P': 6, 'II': 7, 'IA': 8, 'PA': 6, 'A': 7, 'ED': 8, '-': 1}


#### nodes_scale_activity

In [63]:
nodes_scale_activity = {}
ndigits = 5
for hg in HG_desc:
    nodes = hg['nodes']
    scores = hg['values']
    scale_scores = ut.sum_values_per_scale(nodes, scores, items_scales_dict)
    scores_avg = {k: v/scale_arity[k] for k, v in scale_scores.items()}
    avg_sum = sum(scores_avg.values())
    normalized = {k: v/avg_sum for k, v in scores_avg.items()}
    #rounded = {k: round(v, ndigits) for k, v in scale_scores.items()}
    rounded = {k: round(v, ndigits) for k, v in normalized.items()}
    # they sum up to 1 since they come from normalized values over all the nodes (and they are a sum of all nodes' values)
    # this does not mean that the values themselves are normalized
    # convert to list of tuples
    tuples = list(rounded.items())
    # sort by value descending
    tuples_sorted = sorted(tuples, key=lambda x: x[1], reverse=True)
    nodes_scale_activity[hg['row']] = pd.DataFrame(tuples_sorted, columns=["scale", "value"])

In [64]:
nodes_scale_activity.keys()

dict_keys(['ANBP_SYN', 'ANBP_RED', 'ANR_SYN', 'ANR_RED', 'BED_OSFED_SYN', 'BED_OSFED_RED', 'BN_SYN', 'BN_RED'])

In [65]:
with open('tables/nodes_scale_activity_SYNERGY_RANKED_filtered_norm.txt', "w") as f:
    for hg_name, hg_df in nodes_scale_activity.items():
        if hg_name.split('_')[-1] == 'SYN':
            f.write(f"### DataFrame {hg_name}\n\n")
            hg_df = hg_df.merge(scales_desc, on="scale", how="inner")
            f.write(hg_df[['scale', 'desc', 'value']].to_string())
            f.write("\n\n")

with open('tables/nodes_scale_activity_REDUNDANCY_RANKED_filtered_norm.txt', "w") as f:
    for hg_name, hg_df in nodes_scale_activity.items():
        if hg_name.split('_')[-1] == 'RED':
            f.write(f"### DataFrame {hg_name}\n\n")
            hg_df = hg_df.merge(scales_desc, on="scale", how="inner")
            f.write(hg_df[['scale', 'desc', 'value']].to_string())
            f.write("\n\n")

### Visualize the nodes' degree eCDFs

In [66]:
M_norm_SYN = M_norm[M_norm.index.astype(str).str.contains("SYN")].copy()
M_norm_RED = M_norm[M_norm.index.astype(str).str.contains("RED")].copy()

#### Plots (saved to files)

In [67]:
# skip dir creation if it already exist
out_dir_sm = "plots"            
os.makedirs(out_dir_sm, exist_ok=True)

In [68]:
viz.plot_degree_ecdf(M_norm_SYN, show=False, file_name='plots/ecdf_degree_SYN_layers')

In [69]:
viz.plot_degree_ecdf(M_norm_RED, show=False, file_name='plots/ecdf_degree_RED_layers')

In [70]:
viz.plot_per_layer_degree_distributions(M_norm_SYN, bins=30, show=False, file_name='plots/per_layer_nodes_degree_SYN')

In [71]:
viz.plot_per_layer_degree_distributions(M_norm_RED, bins=30, show=False, file_name='plots/per_layer_nodes_degree_RED')

### Overlapping Degree & Participation Coefficient

Let $k_{i,\alpha}$ denote the (possibly weighted) degree of node $i$ in layer $\alpha$, and let $M$ be the total number of layers.  The **overlapping degree** of node $i$ is defined as:

$ O_i = \sum_{\alpha=1}^{M} k_{i,\alpha}.$

The participation coefficient quantifies how the connectivity of a node is distributed across the layers of the multiplex.
It identifies whether a node acts as a **specialist** (active mostly in a single layer: $P_i \approx 0$) or as a **generalist** (active in multiple layers $P_i \approx 1$).

The **participation coefficient** of node $i$ is then given by: $
P_i = \frac{M}{M - 1}
\left(
1 -
\sum_{\alpha=1}^{M}
\left(
\frac{k_{i,\alpha}}{O_i}
\right)^2
\right).
$

This formula measures how evenly the connectivity of node $i$ is distributed across layers.

The term $ \frac{k_{i,\alpha}}{O_i}$ represents the fraction of the total connectivity of node $i$  that occurs in layer $\alpha$.

The sum of squared fractions, $
\sum_{\alpha=1}^{M}
\left(
\frac{k_{i,\alpha}}{O_i}
\right)^2,
$  is large when the connectivity is concentrated in one or few layers, and small when the connectivity is evenly distributed.

The normalization factor $\frac{M}{M - 1}$ ensures that
$
0 \le P_i \le 1.
$


#### Overlapping Degrees + distributions (saved to files)

In [72]:
O_i = ut.compute_overlapping_degree(M_norm, norm = False)  # Series
O_i_norm = ut.compute_overlapping_degree(M_norm, norm = True)  # Series

In [73]:
O_i_SYN = ut.compute_overlapping_degree(M_norm_SYN, norm = False)  # Series
O_i_norm_SYN = ut.compute_overlapping_degree(M_norm_SYN, norm = True)  # Series

O_i_RED = ut.compute_overlapping_degree(M_norm_RED, norm = False)  # Series
O_i_norm_RED = ut.compute_overlapping_degree(M_norm_RED, norm = True)  # Series

In [74]:
viz.plot_overlapping_degree_distribution(O_i, show=False, file_name='plots/overlapping_coeff')
viz.plot_overlapping_degree_distribution(O_i_norm, show=False, file_name='plots/overlapping_coeff_norm')

In [75]:
viz.plot_overlapping_degree_distribution(O_i_SYN, show=False, file_name='plots/overlapping_coeff_SYN')
viz.plot_overlapping_degree_distribution(O_i_norm_SYN, show=False, file_name='plots/overlapping_coeff_norm_SYN')

viz.plot_overlapping_degree_distribution(O_i_RED, show=False, file_name='plots/overlapping_coeff_RED')
viz.plot_overlapping_degree_distribution(O_i_norm_RED, show=False, file_name='plots/overlapping_coeff_norm_RED')

#### Partecipation coefficients + disribution (saved to files)

In [76]:
M_norm_RED[11]

ANBP_RED         0.000000
ANR_RED          0.000000
BED_OSFED_RED    0.037309
BN_RED           0.000000
Name: 11, dtype: float64

In [77]:
partecipation_coeff = ut.compute_participation_coefficient_v2(M_norm)
partecipation_coeff_SYN = ut.compute_participation_coefficient_v2(M_norm_SYN)
partecipation_coeff_RED = ut.compute_participation_coefficient_v2(M_norm_RED)

In [78]:
viz.plot_participation_distribution(partecipation_coeff, show=False, file_name='plots/part_coeff_nodes')
viz.plot_participation_distribution(partecipation_coeff_SYN, show=False, file_name='plots/part_coeff_nodes_SYN')
viz.plot_participation_distribution(partecipation_coeff_RED, show=False, file_name='plots/part_coeff_nodes_RED')

In [79]:
viz.plot_participation_per_node(partecipation_coeff, show=False, file_name='plots/part_coeff_per_node')
viz.plot_participation_per_node(partecipation_coeff_SYN, show=False, file_name='plots/part_coeff_per_node_SYN')
viz.plot_participation_per_node(partecipation_coeff_RED, show=False, file_name='plots/part_coeff_per_node_RED')

In [80]:
top_specialists = partecipation_coeff.sort_values().head(9).to_frame(name='value')
top_specialists_SYN = partecipation_coeff_SYN.sort_values().head(9).to_frame(name='value')
top_specialists_RED = partecipation_coeff_RED.sort_values().head(9).to_frame(name='value')

In [81]:
# ALL
top_specialists_RED = partecipation_coeff_RED.sort_values().to_frame(name='value')
#top_specialists_RED = top_specialists_RED[top_specialists_RED['value'] > 0]
top_specialists_RED['desc'] = top_specialists_RED.apply(lambda row: nodes_items_dict[row.name], axis=1)

In [82]:
top_specialists['desc'] = top_specialists.apply(lambda row: nodes_items_dict[row.name], axis=1)
top_specialists_SYN['desc'] = top_specialists_SYN.apply(lambda row: nodes_items_dict[row.name], axis=1)
top_specialists_RED['desc'] = top_specialists_RED.apply(lambda row: nodes_items_dict[row.name], axis=1)
top_specialists

,value,desc
node,,
32,0.573907,I am preoccupied with the desire to be thinner.
49,0.584611,"If I gain a pound, I worry that I will keep ga..."
11,0.597472,I feel extremely guilty after overeating.
16,0.627119,I am terrified of gaining weight.
64,0.645210,"When I am upset, I worry that I will start eat..."
46,0.648351,I eat moderately in front of others and stuff ...
29,0.655130,"As a child, I tried very hard to avoid disappo..."
89,0.684276,I know there are people who love me.
28,0.718067,I have gone on eating binges where I felt that...


In [83]:
top_specialists_SYN

,value,desc
node,,
64,0.752744,"When I am upset, I worry that I will start eat..."
29,0.764319,"As a child, I tried very hard to avoid disappo..."
38,0.797574,I think about bingeing (overeating).
89,0.798322,I know there are people who love me.
28,0.828530,I have gone on eating binges where I felt that...
6,0.838090,I wish that I could be younger.
50,0.840397,I feel that I am a valuable person.
53,0.856540,I have the thought of trying to vomit in order...
46,0.857413,I eat moderately in front of others and stuff ...


In [84]:
top_specialists_RED[0:20]

,value,desc
node,,
1,0.000000,I eat sweets and carbohydrates without feeling...
11,0.000000,I feel extremely guilty after overeating.
22,0.000000,I would rather be an adult than a child.
15,0.000000,I am open about my feelings.
26,0.000000,I can clearly identify what emotion I am feeling.
41,0.000000,I have a low opinion of myself.
34,0.000000,I have trouble expressing my emotions to others.
37,0.000000,I feel safe with myself.
47,0.000000,I feel bloated after eating a normal meal.


In [85]:
len(top_specialists_RED)

91

## HYPEREDGES PROPERTIES

#### Paper: Multiplex measures for higher‑order networks (2024) - Lotito, Montresor, Battiston 

In [86]:
HGs_SYN = {name: val for name, val in HG_multiplex.items() if "SYN" in name}
HGs_RED = {name: val for name, val in HG_multiplex.items() if "RED" in name}
#print(HG_multiplex.keys())
#print(HGs_SYN.keys())
#print(HGs_RED.keys())

#### Hyperedges: order distribution per layer

In [87]:
order_stats_df_SYN = hnb.hyperedge_order_discrete_stats(HGs_SYN.values(), HGs_SYN.keys())
order_stats_df_RED = hnb.hyperedge_order_discrete_stats(HGs_RED.values(), HGs_RED.keys())

In [88]:
order_stats_df_SYN

,layer,order,count,fraction
0,ANBP_SYN,3,111,0.053212
1,ANBP_SYN,4,1932,0.926174
2,ANBP_SYN,5,43,0.020614
3,ANR_SYN,3,8,0.003612
4,ANR_SYN,4,1850,0.835214
5,ANR_SYN,5,357,0.161174
6,BED_OSFED_SYN,3,190,0.096447
7,BED_OSFED_SYN,4,1769,0.897970
8,BED_OSFED_SYN,5,11,0.005584
9,BN_SYN,3,4,0.002201


In [89]:
order_stats_df_RED

,layer,order,count,fraction
0,ANBP_RED,3,17,0.566667
1,ANBP_RED,4,2,0.066667
2,ANBP_RED,5,11,0.366667
3,ANR_RED,3,31,0.775000
4,ANR_RED,4,2,0.050000
5,ANR_RED,5,7,0.175000
6,BED_OSFED_RED,3,13,0.162500
7,BED_OSFED_RED,4,19,0.237500
8,BED_OSFED_RED,5,48,0.600000
9,BN_RED,3,13,0.650000


In [90]:
viz.plot_hyperedge_order_discrete(order_stats_df_SYN, show=False, file_name='plots/he_order_distro_SYN_layers')
viz.plot_hyperedge_order_discrete(order_stats_df_RED, show=False, file_name='plots/he_order_distro_RED_layers')

### Hyperedges: overlap

In [91]:
edge_to_layers_SYN, overlap_df_SYN = hnb.hyperedge_overlap(HGs_SYN.values(), HGs_SYN.keys())
edge_to_layers_RED, overlap_df_RED = hnb.hyperedge_overlap(HGs_RED.values(), HGs_RED.keys())

sorted_overlap_df_SYN = overlap_df_SYN.sort_values(by="overlap",ascending=False).reset_index(drop=True)
sorted_overlap_df_RED = overlap_df_RED.sort_values(by="overlap",ascending=False).reset_index(drop=True)

In [92]:
common_hes_SYN_list = [tuple([int(i) for i in e.split('-')]) for e in common_hes_SYN] 
#print(common_hes_SYN_list)
sorted_overlap_SYN_keys = [tuple(sorted(e)) for e in sorted_overlap_df_SYN['nodes']]

#print(set(sorted_overlap_SYN_keys) == set(common_hes_SYN_list))

In [93]:
print(len(common_hes_SYN_list), len(sorted_overlap_SYN_keys))

5949 5949


In [94]:
[a for a in sorted(sorted_overlap_SYN_keys) if a not in sorted(common_hes_SYN_list)]

[]

In [98]:
one = [a for a in sorted(common_hes_SYN_list) if a not in sorted(sorted_overlap_SYN_keys)]
print(one) 

[]


In [99]:
counts_SYN = hnb.he_overlap_description(sorted_overlap_df_SYN)
counts_SYN

All hes are  5949
Hes compairing in more layers are 1271
Order 3
Hes compairing in more layers are 25
-----
Order 4
Hes compairing in more layers are 1230
-----
Order 5
Hes compairing in more layers are 16
-----


{'ALL': Counter({('BN_SYN', 'ANBP_SYN', 'BED_OSFED_SYN', 'ANR_SYN'): 237,
          ('ANBP_SYN', 'BED_OSFED_SYN', 'ANR_SYN'): 189,
          ('ANBP_SYN', 'BED_OSFED_SYN'): 180,
          ('ANBP_SYN', 'ANR_SYN'): 123,
          ('ANR_SYN', 'BED_OSFED_SYN'): 115,
          ('BN_SYN', 'ANBP_SYN', 'ANR_SYN'): 110,
          ('BN_SYN', 'ANR_SYN'): 84,
          ('BN_SYN', 'ANBP_SYN'): 79,
          ('BN_SYN', 'BED_OSFED_SYN'): 59,
          ('BN_SYN', 'ANBP_SYN', 'BED_OSFED_SYN'): 53,
          ('BN_SYN', 'ANR_SYN', 'BED_OSFED_SYN'): 42}),
 3: Counter({('ANBP_SYN', 'BED_OSFED_SYN'): 25}),
 4: Counter({('BN_SYN', 'ANBP_SYN', 'BED_OSFED_SYN', 'ANR_SYN'): 237,
          ('ANBP_SYN', 'BED_OSFED_SYN', 'ANR_SYN'): 189,
          ('ANBP_SYN', 'BED_OSFED_SYN'): 155,
          ('ANBP_SYN', 'ANR_SYN'): 121,
          ('ANR_SYN', 'BED_OSFED_SYN'): 115,
          ('BN_SYN', 'ANBP_SYN', 'ANR_SYN'): 110,
          ('BN_SYN', 'ANBP_SYN'): 75,
          ('BN_SYN', 'ANR_SYN'): 74,
          ('BN_SYN', 'BED_

In [100]:
#print(len(sorted_overlap_df_RED))
#sorted_overlap_df_RED[sorted_overlap_df_RED['order'] == 1]

In [101]:
counts_RED = hnb.he_overlap_description(sorted_overlap_df_RED)
counts_RED

All hes are  135
Hes compairing in more layers are 23
Order 3
Hes compairing in more layers are 16
-----
Order 4
Hes compairing in more layers are 1
-----
Order 5
Hes compairing in more layers are 6
-----


{'ALL': Counter({('BED_OSFED_RED', 'ANBP_RED'): 5,
          ('BED_OSFED_RED', 'BN_RED', 'ANBP_RED'): 4,
          ('BN_RED', 'BED_OSFED_RED', 'ANR_RED', 'ANBP_RED'): 3,
          ('BN_RED', 'ANR_RED'): 3,
          ('BN_RED', 'ANBP_RED'): 3,
          ('BN_RED', 'ANR_RED', 'ANBP_RED'): 2,
          ('BED_OSFED_RED', 'BN_RED'): 1,
          ('ANR_RED', 'ANBP_RED'): 1,
          ('BED_OSFED_RED', 'ANR_RED'): 1}),
 3: Counter({('BED_OSFED_RED', 'ANBP_RED'): 4,
          ('BN_RED', 'BED_OSFED_RED', 'ANR_RED', 'ANBP_RED'): 3,
          ('BN_RED', 'ANR_RED'): 3,
          ('BN_RED', 'ANR_RED', 'ANBP_RED'): 2,
          ('BN_RED', 'ANBP_RED'): 2,
          ('BED_OSFED_RED', 'BN_RED', 'ANBP_RED'): 1,
          ('ANR_RED', 'ANBP_RED'): 1}),
 4: Counter({('BN_RED', 'ANBP_RED'): 1}),
 5: Counter({('BED_OSFED_RED', 'BN_RED', 'ANBP_RED'): 3,
          ('BED_OSFED_RED', 'BN_RED'): 1,
          ('BED_OSFED_RED', 'ANBP_RED'): 1,
          ('BED_OSFED_RED', 'ANR_RED'): 1})}

In [102]:
viz.hyperedges_order_overlap(overlap_df_SYN, show=False, file_name='plots/he_order_overlap_SYN')
viz.hyperedges_order_overlap(overlap_df_RED, show=False, file_name='plots/he_order_overlap_RED')

#### Hyperedge weights distribution ($\Omega$ - distribution)

In [103]:
he_weight_SYN = hnb.hyperedge_weight_distributions(HGs_SYN.values(), HGs_SYN.keys())
he_weight_stats_SYN = (he_weight_SYN.groupby(["layer", "order"])["weight"]
                                    .agg(["count", "mean", "std", "min", "max"])
                                    .reset_index())

he_weight_RED = hnb.hyperedge_weight_distributions(HGs_RED.values(), HGs_RED.keys())
he_weight_stats_RED = (he_weight_RED.groupby(["layer", "order"])["weight"]
                                    .agg(["count", "mean", "std", "min", "max"])
                                    .reset_index())

In [104]:
he_weight_SYN.head()

,layer,edge_id,order,weight
0,ANBP_SYN,3-35-39,3,-0.107673
1,ANBP_SYN,5-29-71,3,-0.268419
2,ANBP_SYN,6-14-35,3,-0.116951
3,ANBP_SYN,6-18-24,3,-0.101958
4,ANBP_SYN,6-22-35,3,-0.102736


In [105]:
len(he_weight_RED[he_weight_RED['layer'] == 'ANR_RED'])

40

In [106]:
print('Order 3, syn', he_weight_SYN.loc[he_weight_SYN["order"] == 3, "weight"].mean())
print('Order 4, syn', he_weight_SYN.loc[he_weight_SYN["order"] == 4, "weight"].mean())
print('Order 5, syn', he_weight_SYN.loc[he_weight_SYN["order"] == 5, "weight"].mean())

print('Order 3, red', he_weight_RED.loc[he_weight_RED["order"] == 3, "weight"].mean())
print('Order 4, red', he_weight_RED.loc[he_weight_RED["order"] == 4, "weight"].mean())
print('Order 5, red', he_weight_RED.loc[he_weight_RED["order"] == 5, "weight"].mean())

Order 3, syn -0.1366397936990645
Order 4, syn -0.7616814565302984
Order 5, syn -1.166374915136896
Order 3, red 0.18219029682755225
Order 4, red 0.1998885249690657
Order 5, red 0.24398982588170762


In [107]:
he_weight_RED.loc[he_weight_RED["order"] == 4]

,layer,edge_id,order,weight
0,ANBP_RED,9-45-55-59,4,0.229581
1,ANBP_RED,12-19-55-62,4,0.296680
48,ANR_RED,2-9-45-59,4,0.236164
49,ANR_RED,31-55-59-62,4,0.270183
70,BED_OSFED_RED,9-19-45-55,4,0.136479
71,BED_OSFED_RED,12-45-59-62,4,0.206044
72,BED_OSFED_RED,16-25-32-49,4,0.222532
73,BED_OSFED_RED,31-45-59-62,4,0.134481
90,BED_OSFED_RED,2-7-11-16,4,0.102134
91,BED_OSFED_RED,2-7-16-49,4,0.121542


In [108]:
viz.plot_weights_by_layer_all_orders(
    he_weight_SYN,
    out_dir="plots/he_weights_syn_order",
    file_prefix="weights_vs_layer",
    file_ext="png",
    show=False,
)

In [109]:
viz.plot_weights_by_layer_all_orders(
    he_weight_RED,
    out_dir="plots/he_weights_red_order",
    file_prefix="weights_vs_layer",
    file_ext="png",
    show=False,
)

In [110]:
viz.plot_weights_by_order_all_layers(
    he_weight_SYN,
    out_dir="plots/he_weights_syn",
    file_prefix="weights_vs_order",
    file_ext="png",
    show=False,
)

In [111]:
viz.plot_weights_by_order_all_layers(
    he_weight_RED,
    out_dir="plots/he_weights_red",
    file_prefix="weights_vs_order",
    file_ext="png",
    show=False,
)

## Which sub_scales work together?

In [112]:
order_keys = ["sum_w", "n_hyperedges", "pattern_arity"]
ascending = (True, False, False)

### SYNERGY

#### Inter sub_scales (patterns' arity > 1)

In [113]:
# items_scales_dict contains the mapping node: sub_scale
scales_together_SYN = {}
for hg_name, hg in HGs_SYN.items():
    scales_together_SYN[hg_name] = {}
    for order in (3, 4, 5):
        # weight are negative: ascending true
        scales_together_SYN[hg_name][order] = hnb.pattern_stats_for_order(hg, 
                                                                          items_scales_dict, 
                                                                          order,
                                                                          sorted_by=order_keys,
                                                                          ascending=ascending)

In [114]:
scales_together_SYN['ANBP_SYN'][5].head()

,order,pattern,pattern_arity,n_hyperedges,sum_w,avg_weight,n_he_norm
0,5,"(ED, ID, LSE, P)",4,2,-2.518483,-1.259241,0.04878
1,5,"(A, B, ED, ID)",4,2,-2.030348,-1.015174,0.04878
2,5,"(B, ED, ID, II)",4,2,-2.016732,-1.008366,0.04878
3,5,"(DT, ED)",2,2,-1.881354,-0.940677,0.04878
4,5,"(DT, ED, ID, P)",4,1,-1.407531,-1.407531,0.02439


In [115]:
#scales_together_5_SYN['ANBP_SYN']

#### Intra sub_scales (patterns' arity == 1)

In [116]:
scales_alone_SYN = {}
for hg_name, hg in HGs_SYN.items():
    scales_alone_SYN[hg_name] = {}
    for order in (3, 4, 5):
        # weight are negative: ascending true
        scales_alone_SYN[hg_name][order] = hnb.pattern_stats_for_order(hg, 
                                                                          items_scales_dict, 
                                                                          order,
                                                                          sorted_by=order_keys,
                                                                          ascending=ascending, 
                                                                          arity=False)

In [125]:
# skip dir creation if it already exist
out_dir_sm = "tables/SYN"            
os.makedirs(out_dir_sm, exist_ok=True)

out_dir_sm = "tables/SYN/3"            
os.makedirs(out_dir_sm, exist_ok=True)
out_dir_sm = "tables/SYN/4"            
os.makedirs(out_dir_sm, exist_ok=True)
out_dir_sm = "tables/SYN/5"            
os.makedirs(out_dir_sm, exist_ok=True)
out_dir_sm = "tables/SYN/ALL"            
os.makedirs(out_dir_sm, exist_ok=True)

out_dir_sm = "tables/RED"            
os.makedirs(out_dir_sm, exist_ok=True)

#### Order 3: FOCUS

In [122]:
scales_together_3_SYN = {}
for layer, table in scales_together_SYN.items():
    table_3 = table[3]
    scales_together_3_SYN[layer] = table_3
    print(layer + ':', len(table_3),'different patterns.')
    
print('--------------------------')

scales_alone_3_SYN = {}
for layer, table in scales_alone_SYN.items():
    table_3 = table[3]
    scales_alone_3_SYN[layer] = table_3
    print(layer + ':', len(table_3),'different patterns.')
    
viz.tables_to_txt_order("tables/SYN/3/scales_together_3_SYN.txt", scales_together_3_SYN)
viz.tables_to_txt_order("tables/SYN/3/scales_alone_3_SYN.txt", scales_alone_3_SYN)

ANBP_SYN: 21 different patterns.
ANR_SYN: 3 different patterns.
BED_OSFED_SYN: 42 different patterns.
BN_SYN: 3 different patterns.
--------------------------
ANBP_SYN: 11 different patterns.
ANR_SYN: 4 different patterns.
BED_OSFED_SYN: 10 different patterns.
BN_SYN: 1 different patterns.


#### Order 4: FOCUS

In [123]:
scales_together_4_SYN = {}
for layer, table in scales_together_SYN.items():
    table_4 = table[4]
    scales_together_4_SYN[layer] = table_4
    print(layer + ':', len(table_4),'different patterns.')
    
print('--------------------------')

scales_alone_4_SYN = {}
for layer, table in scales_alone_SYN.items():
    table_4 = table[4]
    scales_alone_4_SYN[layer] = table_4
    print(layer + ':', len(table_4),'different patterns.')
    
viz.tables_to_txt_order("tables/SYN/4/scales_together_4_SYN.txt", scales_together_4_SYN)
viz.tables_to_txt_order("tables/SYN/4/scales_alone_4_SYN.txt", scales_alone_4_SYN)

ANBP_SYN: 248 different patterns.
ANR_SYN: 232 different patterns.
BED_OSFED_SYN: 244 different patterns.
BN_SYN: 176 different patterns.
--------------------------
ANBP_SYN: 12 different patterns.
ANR_SYN: 12 different patterns.
BED_OSFED_SYN: 12 different patterns.
BN_SYN: 12 different patterns.


#### Order 5: FOCUS

In [124]:
scales_together_5_SYN = {}
for layer, table in scales_together_SYN.items():
    table_5 = table[5]
    scales_together_5_SYN[layer] = table_5
    print(layer + ':', len(table_5),'different patterns.')
    
print('--------------------------')

scales_alone_5_SYN = {}
for layer, table in scales_alone_SYN.items():
    table_5 = table[5]
    scales_alone_5_SYN[layer] = table_5
    print(layer + ':', len(table_5),'different patterns.')

viz.tables_to_txt_order("tables/SYN/5/scales_together_5_SYN.txt", scales_together_5_SYN) 
viz.tables_to_txt_order("tables/SYN/5/scales_alone_5_SYN.txt", scales_alone_5_SYN)

ANBP_SYN: 35 different patterns.
ANR_SYN: 163 different patterns.
BED_OSFED_SYN: 7 different patterns.
BN_SYN: 222 different patterns.
--------------------------
ANBP_SYN: 2 different patterns.
ANR_SYN: 7 different patterns.
BED_OSFED_SYN: 0 different patterns.
BN_SYN: 8 different patterns.


#### One table for ALL ORDERS.

In [127]:
scales_together_ALL_SYN = {}
for hg_name, hg in HGs_SYN.items():
    scales_together_ALL_SYN[hg_name] = hnb.pattern_stats_for_layer(hg, items_scales_dict,
                                            sorted_by=order_keys,
                                            ascending=ascending)
    
for layer, table in scales_together_ALL_SYN.items():
    print(layer + ':', len(table),'different patterns.')
    
# scales_together_ALL_SYN['ANBP_SYN'].head()
viz.tables_to_txt("tables/SYN/ALL/scales_together_ALL_SYN.txt", scales_together_ALL_SYN)
viz.tables_to_txt("tables/SYN/ALL/scales_together_ALL_SYN_top.txt", scales_together_ALL_SYN, top=True)

ANBP_SYN: 270 different patterns.
ANR_SYN: 331 different patterns.
BED_OSFED_SYN: 250 different patterns.
BN_SYN: 315 different patterns.


In [128]:
for layer, table in scales_together_ALL_SYN.items():
    print(layer + ':', table["n_hyperedges"].sum())

ANBP_SYN: 1474
ANR_SYN: 1664
BED_OSFED_SYN: 1420
BN_SYN: 1191


#### Observe the one - sub_scaled patters.

In [129]:
scales_alone_ALL_SYN = {}
for hg_name, hg in HGs_SYN.items():
    scales_alone_ALL_SYN[hg_name] = hnb.pattern_stats_for_layer(hg, items_scales_dict,
                                            sorted_by=order_keys,
                                            ascending=ascending,
                                            arity = False)

viz.tables_to_txt("tables/SYN/ALL/scales_alone_ALL_SYN.txt", scales_alone_ALL_SYN)

In [130]:
for layer, table in scales_alone_ALL_SYN.items():
    print(layer + ':', table["n_hyperedges"].sum())

ANBP_SYN: 612
ANR_SYN: 551
BED_OSFED_SYN: 550
BN_SYN: 626


#### ALL ORDERS (3 EXCLUDED)

In [131]:
out_dir_sm = "tables/SYN/minus3"            
os.makedirs(out_dir_sm, exist_ok=True)

In [132]:
scales_together_ALL_minus3_SYN = {}
for hg_name, hg in HGs_SYN.items():
    scales_together_ALL_minus3_SYN[hg_name] = hnb.pattern_stats_for_layer(hg, items_scales_dict,
                                            exclude_order=True,
                                            excluded_orders=[3],                       
                                            sorted_by=order_keys,
                                            ascending=ascending)
    
for layer, table in scales_together_ALL_minus3_SYN.items():
    print(layer + ':', len(table),'different patterns.')
    
viz.tables_to_txt("tables/SYN/minus3/scales_together_ALL_minus3_SYN.txt", scales_together_ALL_minus3_SYN)
viz.tables_to_txt("tables/SYN/minus3/scales_together_ALL_minus3_SYN_top.txt", scales_together_ALL_minus3_SYN, top=True)

ANBP_SYN: 266 different patterns.
ANR_SYN: 330 different patterns.
BED_OSFED_SYN: 245 different patterns.
BN_SYN: 315 different patterns.


#### Observe the one - sub_scaled patters.

In [133]:
scales_alone_ALL_minus3_SYN = {}
for hg_name, hg in HGs_SYN.items():
    scales_alone_ALL_minus3_SYN[hg_name] = hnb.pattern_stats_for_layer(hg, items_scales_dict,
                                            exclude_order=True,
                                            excluded_orders=[3],                          
                                            sorted_by=order_keys,
                                            ascending=ascending,
                                            arity = False)

viz.tables_to_txt("tables/SYN/minus3/scales_alone_ALL_minus3_SYN.txt", scales_alone_ALL_minus3_SYN)

### REDUNDANCY

In [134]:
order_keys = ["sum_w", "n_hyperedges", "pattern_arity"]
ascending = (False, False, False)

#### Inter sub_scales (patterns' arity > 1)

In [135]:
scales_together_RED = {}
for hg_name, hg in HGs_RED.items():
    scales_together_RED[hg_name] = {}
    for order in (3, 4, 5):
        scales_together_RED[hg_name][order] = hnb.pattern_stats_for_order(hg, 
                                                                          items_scales_dict, 
                                                                          order,
                                                                          sorted_by=order_keys,
                                                                          ascending=ascending)

#### Intra sub_scales (patterns' arity == 1)

In [136]:
scales_alone_RED = {}
for hg_name, hg in HGs_RED.items():
    scales_alone_RED[hg_name] = {}
    for order in (3, 4, 5):
        # weight are negative: ascending true
        scales_alone_RED[hg_name][order] = hnb.pattern_stats_for_order(hg, 
                                                                          items_scales_dict, 
                                                                          order,
                                                                          sorted_by=order_keys,
                                                                          ascending=ascending, 
                                                                          arity=False)

#### One table for ALL ORDERS.

In [137]:
scales_together_ALL_RED = {}
for hg_name, hg in HGs_RED.items():
    scales_together_ALL_RED[hg_name] = hnb.pattern_stats_for_layer(hg, items_scales_dict,
                                            sorted_by=order_keys,
                                            ascending=ascending)

In [138]:
for layer, table in scales_together_ALL_RED.items():
    print(layer + ':', len(table),'different patterns.')

ANBP_RED: 1 different patterns.
ANR_RED: 2 different patterns.
BED_OSFED_RED: 6 different patterns.
BN_RED: 0 different patterns.


In [139]:
viz.tables_to_txt("tables/RED/scales_together_ALL_RED.txt", scales_together_ALL_RED)

#### Observe the one - sub_scaled patterns.

In [140]:
scales_alone_ALL_RED = {}
for hg_name, hg in HGs_RED.items():
    scales_alone_ALL_RED[hg_name] = hnb.pattern_stats_for_layer(hg, items_scales_dict,
                                            sorted_by=order_keys,
                                            ascending=ascending,
                                            arity = False)

In [141]:
viz.tables_to_txt("tables/RED/scales_alone_ALL_RED.txt", scales_alone_ALL_RED)

# -----------------------------------------------------------------------------------------------------------------

#### Patterns - to - hes - list : SYN

In [142]:
scales_hes_SYN = {}
for hg_name, hg in HGs_SYN.items():
    scales_hes_SYN[hg_name] = hnb.pattern_to_hes_list(hg, items_scales_dict, 'weight', True)

In [143]:
for layer, D in scales_hes_SYN.items():
    print(layer)
    for k, val in D.items():
        print(k, len(val))
        print(val)
        print('-----------------')
    print('\n')

ANBP_SYN
frozenset({'MF'}) 59
[([6, 35, 39, 48], -0.6989352646854528), ([3, 35, 39, 48], -0.6441795272257202), ([14, 35, 48, 58], -0.6355424202228441), ([3, 22, 35, 48], -0.6310068752179987), ([3, 14, 35, 39], -0.6301539161109062), ([35, 39, 48, 58], -0.6234347356817054), ([6, 35, 48, 58], -0.6078725474161217), ([3, 14, 35, 58], -0.6046850892365256), ([3, 35, 48, 58], -0.5958617712186438), ([6, 14, 35, 58], -0.5884832497479735), ([6, 14, 35, 39], -0.5680194467001627), ([22, 35, 48, 58], -0.5482270562953921), ([3, 35, 39, 58], -0.5461989308062059), ([3, 6, 35, 58], -0.5428856937865163), ([14, 22, 35, 48], -0.5402764596965106), ([14, 22, 35, 58], -0.5203283844524691), ([14, 35, 39, 48], -0.5145468880743529), ([3, 22, 35, 39], -0.5125488180146647), ([22, 35, 39, 48], -0.5005977644930226), ([3, 14, 22, 35], -0.4873212907188282), ([3, 14, 48, 58], -0.4843490396326899), ([14, 35, 39, 58], -0.4693441340582183), ([3, 39, 48, 58], -0.4673347786111996), ([3, 22, 35, 58], -0.4610389536438184), ([

#### Patterns - to - hes - list : RED

In [144]:
scales_hes_RED = {}
for hg_name, hg in HGs_RED.items():
    scales_hes_RED[hg_name] = hnb.pattern_to_hes_list(hg, items_scales_dict, 'weight', False)

In [145]:
for layer, D in scales_hes_RED.items():
    print(layer)
    for k, val in D.items():
        print(k, len(val))
        print(val)
        print('-----------------')
    print('\n')

ANBP_RED
frozenset({'BD'}) 24
[([31, 45, 55, 59, 62], 0.3631796788126422), ([12, 19, 55, 62], 0.296679539545309), ([9, 45, 55], 0.2810383363406146), ([45, 55, 62], 0.261841682577705), ([9, 45, 55, 59], 0.2295813205180543), ([12, 19, 45, 59, 62], 0.2191317643260752), ([9, 55, 62], 0.2021145460497262), ([9, 45, 62], 0.1963873877675297), ([9, 12, 19, 31, 55], 0.1939259218755147), ([19, 55, 62], 0.1875858541347632), ([2, 12, 19], 0.1802370336264438), ([55, 59, 62], 0.1789274033280428), ([2, 12, 31, 55, 62], 0.169297250329194), ([31, 55, 62], 0.1543661865714227), ([19, 45, 62], 0.1524056163274174), ([9, 55, 59], 0.1491861635656519), ([2, 12, 45, 55, 62], 0.1467427589803875), ([31, 45, 55], 0.1456458760021721), ([12, 55, 62], 0.1441512122864097), ([9, 19, 31, 45, 55], 0.1433750031927232), ([9, 12, 19, 45, 62], 0.1385652261793168), ([9, 12, 19, 45, 55], 0.1326167951041057), ([2, 9, 19, 45, 62], 0.1270093143209827), ([2, 9, 12, 45, 55], 0.116993289111388)]
-----------------
frozenset({'DT', 'B